<h1 style="border-bottom: 3px solid #4CAF50; padding-bottom: 8px;">
  📦 00 — Project Setup
</h1>

<table style="margin-top: 12px; font-size: 14px;">
  <tr><td style="padding-right: 14px;"><b>Objective</b></td><td>Bootstrap the development environment — install dependencies, configure paths, create directory structure, authenticate with Kaggle, download raw dataset, and verify file inventory integrity.</td></tr>
  <tr><td><b>Dataset</b></td><td><a href="https://www.kaggle.com/competitions/petfinder-adoption-prediction">PetFinder Adoption Prediction</a></td></tr>
  <tr><td><b>Dependencies</b></td><td>None (entry-point notebook)</td></tr>
  <tr><td><b>Artifacts</b></td><td>Raw dataset in <code>data/raw/</code> · Full directory scaffold</td></tr>
  <tr><td><b>Author</b></td><td>Pedro Markovicz</td></tr>
</table>

---
## Step 1 · Install Dependencies

Install the project as an **editable package** from `pyproject.toml`, making `src/adoption_accelerator` importable across all notebooks.

In [1]:
import subprocess, sys, importlib, pathlib

# Ensure src/ is on sys.path so the current kernel session can import the package
# even before (or regardless of) .pth file processing at startup.
_src = (pathlib.Path.cwd().parent / "src").resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Install the package as editable so it is properly registered for future kernel startups.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", "..", "--quiet"],
    stdout=subprocess.DEVNULL,
)
importlib.invalidate_caches()
print("Package installed successfully.")


Package installed successfully.


---
## Step 2 · Import Libraries & Version Check

Import core libraries and print versions for **reproducibility logging**.

In [2]:
import sys
import pathlib

import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import plotly
from IPython.display import display, Markdown

# ── Version report ──────────────────────────────────────────────────
libs = [pd, np, matplotlib, sns, plotly]
versions = " | ".join(f"**{lib.__name__}** `{lib.__version__}`" for lib in libs)

display(Markdown(
    f"**Python** `{sys.version.split()[0]}`  &ensp;·&ensp; {versions}\n\n"
    f"**CWD** `{pathlib.Path.cwd()}`"
))

**Python** `3.13.5`  &ensp;·&ensp; **pandas** `3.0.1` | **numpy** `2.4.2` | **matplotlib** `3.10.8` | **seaborn** `0.13.2` | **plotly** `6.5.2`

**CWD** `c:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\notebooks`

---
## Step 3 · Load Configuration

Import path constants from `src/adoption_accelerator/config.py`.  
All paths originate from the centralized config — **no hardcoded paths in the notebook**.

In [3]:
from adoption_accelerator.config import (
    PROJECT_ROOT, SEED,
    DATA_DIR, DATA_RAW, DATA_CLEANED, DATA_FEATURES, DATA_SUBMISSIONS,
    REPORTS_DIR, REPORTS_FIGURES, REPORTS_METRICS,
    ARTIFACTS_DIR, ARTIFACTS_MODELS,
    KAGGLE_COMPETITION, EXPECTED_FILE_COUNTS,
    RAW_TRAIN_CSV, RAW_TEST_CSV,
    REF_BREED_LABELS, REF_COLOR_LABELS, REF_STATE_LABELS,
    RAW_TRAIN_IMAGES, RAW_TEST_IMAGES,
    RAW_TRAIN_METADATA, RAW_TEST_METADATA,
    RAW_TRAIN_SENTIMENT, RAW_TEST_SENTIMENT,
)

display(Markdown(
    f"| Variable | Value |\n|---|---|\n"
    f"| `PROJECT_ROOT` | `{PROJECT_ROOT}` |\n"
    f"| `DATA_RAW` | `{DATA_RAW}` |\n"
    f"| `DATA_CLEANED` | `{DATA_CLEANED}` |\n"
    f"| `DATA_FEATURES` | `{DATA_FEATURES}` |\n"
    f"| `DATA_SUBMISSIONS` | `{DATA_SUBMISSIONS}` |\n"
    f"| `REPORTS_DIR` | `{REPORTS_DIR}` |\n"
    f"| `ARTIFACTS_MODELS` | `{ARTIFACTS_MODELS}` |\n"
    f"| `SEED` | `{SEED}` |"
))

| Variable | Value |
|---|---|
| `PROJECT_ROOT` | `C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator` |
| `DATA_RAW` | `C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\raw` |
| `DATA_CLEANED` | `C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\cleaned` |
| `DATA_FEATURES` | `C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features` |
| `DATA_SUBMISSIONS` | `C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\submissions` |
| `REPORTS_DIR` | `C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports` |
| `ARTIFACTS_MODELS` | `C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\artifacts\models` |
| `SEED` | `42` |

---
## Step 4 · Create Directory Structure

Ensure all project directories exist — **idempotent** (safe to re-run).

In [4]:
from adoption_accelerator.utils.paths import ensure_directories

dirs = ensure_directories()

rows = "\n".join(
    f"| ✅ | `{d.relative_to(PROJECT_ROOT)}` |" for d in dirs
)
display(Markdown(f"| Status | Directory |\n|:---:|---|\n{rows}"))

| Status | Directory |
|:---:|---|
| ✅ | `data\raw` |
| ✅ | `data\cleaned` |
| ✅ | `data\features` |
| ✅ | `data\submissions` |
| ✅ | `reports\figures` |
| ✅ | `reports\metrics` |
| ✅ | `artifacts\models` |

---
## Step 5 · Global Defaults

Set random seed (`SEED = 42`), pandas display options, matplotlib/seaborn theme, and logging level.

In [5]:
import logging
from adoption_accelerator.utils.logging import setup_logging

# ── Reproducibility ─────────────────────────────────────────────────
np.random.seed(SEED)

# ── Pandas display ──────────────────────────────────────────────────
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Matplotlib / Seaborn ────────────────────────────────────────────
matplotlib.rcParams["figure.figsize"] = (10, 6)
matplotlib.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid", palette="muted")

# ── Logging ─────────────────────────────────────────────────────────
log = setup_logging(logging.INFO)
log.info("Global settings applied · SEED=%d", SEED)

display(Markdown(
    "| Setting | Value |\n|---|---|\n"
    f"| Random seed | `{SEED}` |\n"
    "| Pandas max columns | `60` |\n"
    "| Figure size | `(10, 6)` |\n"
    "| Figure DPI | `100` |\n"
    "| Seaborn theme | `whitegrid` |\n"
    f"| Logging level | `INFO` |"
))

21:58:31  INFO      Global settings applied · SEED=42


| Setting | Value |
|---|---|
| Random seed | `42` |
| Pandas max columns | `60` |
| Figure size | `(10, 6)` |
| Figure DPI | `100` |
| Seaborn theme | `whitegrid` |
| Logging level | `INFO` |

---
## Step 6 · Verify Kaggle Credentials

Assert that `~/.kaggle/kaggle.json` exists and is readable.

In [9]:
from dotenv import load_dotenv
import os
from pathlib import Path
import json

# Load .env variables
load_dotenv()

# Get credentials
username = os.getenv("KAGGLE_USERNAME")
key = os.getenv("KAGGLE_API_TOKEN")

assert username and key, "Missing Kaggle credentials in .env file"

# Create ~/.kaggle directory
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

# Create kaggle.json
kaggle_json = kaggle_dir / "kaggle.json"

creds = {
    "username": username,
    "key": key
}

kaggle_json.write_text(json.dumps(creds))

# Secure permissions (important on Linux/Mac)
try:
    kaggle_json.chmod(0o600)
except:
    pass

print("✅ Kaggle credentials configured!")

✅ Kaggle credentials configured!


---
## Step 7 · Download Dataset

Download `petfinder-adoption-prediction` from the Kaggle API into `data/raw/`.  
**Idempotent** — skips download if the ZIP file already exists.

In [10]:
import subprocess

ZIP = DATA_RAW / f"{KAGGLE_COMPETITION}.zip"

if not ZIP.exists():
    log.info("Downloading dataset from Kaggle...")
    subprocess.run(
        ["kaggle", "competitions", "download",
         "-c", KAGGLE_COMPETITION,
         "-p", str(DATA_RAW)],
        check=True,
    )
    display(Markdown(f"✅ **Downloaded** → `{ZIP.relative_to(PROJECT_ROOT)}`"))
else:
    display(Markdown(f"✅ **ZIP already present** → `{ZIP.relative_to(PROJECT_ROOT)}`"))

22:03:29  INFO      Downloading dataset from Kaggle...


✅ **Downloaded** → `data\raw\petfinder-adoption-prediction.zip`

---
## Step 8 · Extract Dataset

Unzip into `data/raw/` and organize files into the expected subdirectory layout.

In [11]:
import zipfile

if ZIP.exists():
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(DATA_RAW)
        display(Markdown(f"✅ **Extracted** `{len(zf.namelist())}` items → `{DATA_RAW.relative_to(PROJECT_ROOT)}`"))
else:
    display(Markdown("⚠️ ZIP file not found — cannot extract."))

✅ **Extracted** `163871` items → `data\raw`

---
## Step 9 · File Inventory

List all files in `data/raw/` with **counts per directory** and aggregated file sizes.

In [12]:
from collections import defaultdict

dir_stats = defaultdict(lambda: {"count": 0, "size_bytes": 0})

for p in sorted(DATA_RAW.rglob("*")):
    if p.is_file() and p.name != f"{KAGGLE_COMPETITION}.zip":
        rel_dir = p.parent.relative_to(DATA_RAW)
        key = str(rel_dir) if str(rel_dir) != "." else "(root)"
        dir_stats[key]["count"] += 1
        dir_stats[key]["size_bytes"] += p.stat().st_size

# Build summary table
rows = []
total_files = 0
total_size = 0
for dirname, stats in sorted(dir_stats.items()):
    size_mb = stats["size_bytes"] / 1_048_576
    rows.append(f"| `{dirname}` | {stats['count']:,} | {size_mb:,.1f} MB |")
    total_files += stats["count"]
    total_size += stats["size_bytes"]

rows.append(f"| **TOTAL** | **{total_files:,}** | **{total_size / 1_048_576:,.1f} MB** |")

display(Markdown(
    "| Directory | Files | Size |\n|---|---:|---:|\n" + "\n".join(rows)
))

| Directory | Files | Size |
|---|---:|---:|
| `(root)` | 10 | 0.0 MB |
| `test` | 2 | 1.6 MB |
| `test_images` | 14,465 | 373.8 MB |
| `test_metadata` | 14,465 | 73.6 MB |
| `test_sentiment` | 3,865 | 15.7 MB |
| `train` | 1 | 6.4 MB |
| `train_images` | 58,311 | 1,531.7 MB |
| `train_metadata` | 58,311 | 298.3 MB |
| `train_sentiment` | 14,442 | 64.4 MB |
| **TOTAL** | **163,872** | **2,365.7 MB** |

---
## Step 10 · Validation Gate

Assert-based checkpoints with **pass/fail** results.  
All gates must pass for the setup to be considered complete.

| Gate | Assertion |
|------|-----------|
| `G00-1` | `data/raw/train/train.csv` exists |
| `G00-2` | `data/raw/test/test.csv` exists |
| `G00-3` | File count in `train_images/` = 58,311 |
| `G00-4` | File count in `test_images/` = 14,465 |
| `G00-5` | File count in `train_metadata/` = 58,311 |
| `G00-6` | File count in `test_metadata/` = 14,465 |
| `G00-7` | File count in `train_sentiment/` = 14,442 |
| `G00-8` | File count in `test_sentiment/` = 3,865 |
| `G00-9` | Reference CSVs exist |
| `G00-10` | `src/adoption_accelerator` is importable |

In [13]:
def _count_files(directory):
    """Count files in a directory (non-recursive)."""
    return sum(1 for f in directory.iterdir() if f.is_file())


results = []

# ── G00-1: train.csv exists ────────────────────────────────────────
ok = RAW_TRAIN_CSV.exists()
results.append(("G00-1", "`train/train.csv` exists", ok))

# ── G00-2: test.csv exists ─────────────────────────────────────────
ok = RAW_TEST_CSV.exists()
results.append(("G00-2", "`test/test.csv` exists", ok))

# ── G00-3 to G00-8: File counts ────────────────────────────────────
dir_map = {
    "G00-3": ("train_images", RAW_TRAIN_IMAGES),
    "G00-4": ("test_images",  RAW_TEST_IMAGES),
    "G00-5": ("train_metadata", RAW_TRAIN_METADATA),
    "G00-6": ("test_metadata",  RAW_TEST_METADATA),
    "G00-7": ("train_sentiment", RAW_TRAIN_SENTIMENT),
    "G00-8": ("test_sentiment",  RAW_TEST_SENTIMENT),
}

for gate_id, (name, path) in dir_map.items():
    expected = EXPECTED_FILE_COUNTS[name]
    actual = _count_files(path)
    ok = actual == expected
    results.append((gate_id, f"`{name}/` = {actual:,} (expected {expected:,})", ok))

# ── G00-9: Reference CSVs exist ────────────────────────────────────
ref_ok = all(p.exists() for p in [REF_BREED_LABELS, REF_COLOR_LABELS, REF_STATE_LABELS])
results.append(("G00-9", "Reference CSVs (`breed`, `color`, `state`) exist", ref_ok))

# ── G00-10: Package importable ──────────────────────────────────────
try:
    import adoption_accelerator
    pkg_ok = True
except ImportError:
    pkg_ok = False
results.append(("G00-10", "`adoption_accelerator` is importable", pkg_ok))

# ── Display results ─────────────────────────────────────────────────
rows = "\n".join(
    f"| {gate} | {desc} | {'✅ PASS' if ok else '❌ FAIL'} |"
    for gate, desc, ok in results
)
display(Markdown(f"| Gate | Check | Result |\n|---|---|:---:|\n{rows}"))

# ── Assert all pass ─────────────────────────────────────────────────
failed = [(g, d) for g, d, ok in results if not ok]
assert not failed, (
    f"❌ {len(failed)} validation gate(s) FAILED:\n"
    + "\n".join(f"  • {g}: {d}" for g, d in failed)
)

display(Markdown("### ✅ All validation gates passed."))

| Gate | Check | Result |
|---|---|:---:|
| G00-1 | `train/train.csv` exists | ✅ PASS |
| G00-2 | `test/test.csv` exists | ✅ PASS |
| G00-3 | `train_images/` = 58,311 (expected 58,311) | ✅ PASS |
| G00-4 | `test_images/` = 14,465 (expected 14,465) | ✅ PASS |
| G00-5 | `train_metadata/` = 58,311 (expected 58,311) | ✅ PASS |
| G00-6 | `test_metadata/` = 14,465 (expected 14,465) | ✅ PASS |
| G00-7 | `train_sentiment/` = 14,442 (expected 14,442) | ✅ PASS |
| G00-8 | `test_sentiment/` = 3,865 (expected 3,865) | ✅ PASS |
| G00-9 | Reference CSVs (`breed`, `color`, `state`) exist | ✅ PASS |
| G00-10 | `adoption_accelerator` is importable | ✅ PASS |

### ✅ All validation gates passed.

---

<h2 style="border-top: 2px solid #4CAF50; padding-top: 12px;">Summary</h2>

| Item | Status |
|------|:------:|
| Dependencies installed | ✅ |
| `src/adoption_accelerator` importable | ✅ |
| Directory scaffold created | ✅ |
| Kaggle credentials verified | ✅ |
| Dataset downloaded & extracted | ✅ |
| File inventory verified | ✅ |
| All validation gates passed | ✅ |

**Next →** `01_data_ingestion.ipynb`